# Análisis Exploratorio y Visualización de Datos

Este documento detalla el proceso de exploración visual del conjunto de datos **NSL-KDD**. Este dataset es una referencia estándar para el desarrollo de sistemas de detección de intrusiones (IDS) en entornos de red, ya que clasifica el tráfico entre conexiones normales y diferentes tipos de ataques informáticos.

NSL-KDD soluciona varios problemas de redundancia presentes en el antiguo KDD’99, permitiendo que los modelos de aprendizaje no se sesguen hacia registros repetitivos.

### Fuente de los datos
Los archivos originales se encuentran alojados en Kaggle: **https://www.kaggle.com/datasets/hassan06/nslkdd**.

### Estructura del conjunto de datos
- **KDDTrain+.txt**: Información para el entrenamiento del modelo (incluye tipo de ataque y dificultad).
- **KDDTest+.txt**: Información para la validación final del modelo.
- **Versiones reducidas**: Archivos con el sufijo `_20Percent` para pruebas rápidas.

## 1. Carga de Herramientas y Librerías
Iniciamos importando los paquetes necesarios para la gestión de tablas, generación de gráficos y codificación de variables.

In [5]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from pandas.plotting import scatter_matrix

%matplotlib inline

## 2. Preparación de los Archivos
Dado que los archivos originales no incluyen cabeceras, definimos manualmente el nombre de las 43 columnas y transformamos el formato TXT a CSV para facilitar su uso posterior.

In [6]:
# Listado completo de atributos según la documentación del dataset
NOMBRES_COLUMNAS = [
    "duration", "protocol_type", "service", "flag", "src_bytes", "dst_bytes",
    "land", "wrong_fragment", "urgent", "hot", "num_failed_logins",
    "logged_in", "num_compromised", "root_shell", "su_attempted",
    "num_root", "num_file_creations", "num_shells", "num_access_files",
    "num_outbound_cmds", "is_host_login", "is_guest_login", "count",
    "srv_count", "serror_rate", "srv_serror_rate", "rerror_rate",
    "srv_rerror_rate", "same_srv_rate", "diff_srv_rate",
    "srv_diff_host_rate", "dst_host_count", "dst_host_srv_count",
    "dst_host_same_srv_rate", "dst_host_diff_srv_rate",
    "dst_host_same_src_port_rate", "dst_host_srv_diff_host_rate",
    "dst_host_serror_rate", "dst_host_srv_serror_rate",
    "dst_host_rerror_rate", "dst_host_srv_rerror_rate",
    "class", "difficulty"
]

# Lectura de ficheros planos
df_entrenamiento = pd.read_csv("KDDTrain+.txt", header=None, names=NOMBRES_COLUMNAS)
df_prueba = pd.read_csv("KDDTest+.txt", header=None, names=NOMBRES_COLUMNAS)

# Almacenamiento en formato CSV
df_entrenamiento.to_csv("KDDTrain+.csv", index=False)
df_prueba.to_csv("KDDTest+.csv", index=False)

## 3. Inspección Inicial de los Datos
Revisamos una muestra de los registros para comprender la naturaleza de las variables (numéricas y categóricas).

In [7]:
df_entrenamiento.head()

,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,class,difficulty
0,0,tcp,ftp_data,SF,491,0,0,0,0,0,...,0.17,0.03,0.17,0.00,0.00,0.00,0.05,0.00,normal,20
1,0,udp,other,SF,146,0,0,0,0,0,...,0.00,0.60,0.88,0.00,0.00,0.00,0.00,0.00,normal,15
2,0,tcp,private,S0,0,0,0,0,0,0,...,0.10,0.05,0.00,0.00,1.00,1.00,0.00,0.00,neptune,19
3,0,tcp,http,SF,232,8153,0,0,0,0,...,1.00,0.00,0.03,0.04,0.03,0.01,0.00,0.01,normal,21
4,0,tcp,http,SF,199,420,0,0,0,0,...,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,normal,21


In [8]:
print(f"Dimensiones entrenamiento: {df_entrenamiento.shape}")
print(f"Dimensiones prueba: {df_prueba.shape}")

Dimensiones entrenamiento: (125973, 43)
Dimensiones prueba: (22544, 43)


### Perfilado del Dataset
Analizamos los tipos de datos asignados automáticamente y verificamos si existen valores ausentes.

In [9]:
df_entrenamiento.info()

<class 'pandas.DataFrame'>
RangeIndex: 125973 entries, 0 to 125972
Data columns (total 43 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   duration                     125973 non-null  int64  
 1   protocol_type                125973 non-null  str    
 2   service                      125973 non-null  str    
 3   flag                         125973 non-null  str    
 4   src_bytes                    125973 non-null  int64  
 5   dst_bytes                    125973 non-null  int64  
 6   land                         125973 non-null  int64  
 7   wrong_fragment               125973 non-null  int64  
 8   urgent                       125973 non-null  int64  
 9   hot                          125973 non-null  int64  
 10  num_failed_logins            125973 non-null  int64  
 11  logged_in                    125973 non-null  int64  
 12  num_compromised              125973 non-null  int64  
 13  root_shell

In [10]:
# Resumen estadístico de las columnas numéricas
df_entrenamiento.describe()

,duration,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,num_failed_logins,logged_in,num_compromised,...,dst_host_srv_count,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,difficulty
count,125973.00000,1.259730e+05,1.259730e+05,125973.000000,125973.000000,125973.000000,125973.000000,125973.000000,125973.000000,125973.000000,...,125973.000000,125973.000000,125973.000000,125973.000000,125973.000000,125973.000000,125973.000000,125973.000000,125973.000000,125973.000000
mean,287.14465,4.556674e+04,1.977911e+04,0.000198,0.022687,0.000111,0.204409,0.001222,0.395736,0.279250,...,115.653005,0.521242,0.082951,0.148379,0.032542,0.284452,0.278485,0.118832,0.120240,19.504060
std,2604.51531,5.870331e+06,4.021269e+06,0.014086,0.253530,0.014366,2.149968,0.045239,0.489010,23.942042,...,110.702741,0.448949,0.188922,0.308997,0.112564,0.444784,0.445669,0.306557,0.319459,2.291503
min,0.00000,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.00000,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,10.000000,0.050000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,18.000000
50%,0.00000,4.400000e+01,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,63.000000,0.510000,0.020000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,20.000000
75%,0.00000,2.760000e+02,5.160000e+02,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,...,255.000000,1.000000,0.070000,0.060000,0.020000,1.000000,1.000000,0.000000,0.000000,21.000000
max,42908.00000,1.379964e+09,1.309937e+09,1.000000,3.000000,3.000000,77.000000,5.000000,1.000000,7479.000000,...,255.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,21.000000


## 4. Transformación de Atributos Categóricos
Para poder realizar cálculos matemáticos y visualizaciones de correlación, convertimos las variables de texto en valores numéricos mediante codificación de etiquetas (*Label Encoding*).

In [11]:
codificador = LabelEncoder()
columnas_objeto = ['protocol_type', 'service', 'flag', 'class']

for col in columnas_objeto:
    df_entrenamiento[col] = codificador.fit_transform(df_entrenamiento[col])

df_entrenamiento.head()

,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,class,difficulty
0,0,1,20,9,491,0,0,0,0,0,...,0.17,0.03,0.17,0.00,0.00,0.00,0.05,0.00,11,20
1,0,2,44,9,146,0,0,0,0,0,...,0.00,0.60,0.88,0.00,0.00,0.00,0.00,0.00,11,15
2,0,1,49,5,0,0,0,0,0,0,...,0.10,0.05,0.00,0.00,1.00,1.00,0.00,0.00,9,19
3,0,1,24,9,232,8153,0,0,0,0,...,1.00,0.00,0.03,0.04,0.03,0.01,0.00,0.01,11,21
4,0,1,24,9,199,420,0,0,0,0,...,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,11,21


## 5. Análisis de Relaciones (Correlación)
Evaluamos qué variables tienen una relación más fuerte con nuestra columna objetivo (`class`). Esto ayuda a identificar qué características son determinantes para detectar un ataque.

In [12]:
# Obtener la correlación lineal respecto a la clase, ordenando de mayor a menor impacto
correlaciones = df_entrenamiento.corr()["class"].sort_values(ascending=False)
print(correlaciones)

class                          1.000000
srv_count                      0.310819
wrong_fragment                 0.304125
dst_host_diff_srv_rate         0.295042
same_srv_rate                  0.258357
diff_srv_rate                  0.228557
dst_host_same_src_port_rate    0.202229
flag                           0.170749
dst_host_srv_count             0.138848
protocol_type                  0.135203
duration                       0.134590
is_guest_login                 0.109112
dst_host_rerror_rate           0.108057
dst_host_srv_rerror_rate       0.106217
logged_in                      0.104056
rerror_rate                    0.099397
hot                            0.098611
srv_rerror_rate                0.098105
dst_host_same_srv_rate         0.073548
count                          0.060380
dst_host_count                 0.053869
src_bytes                      0.011617
dst_bytes                      0.007600
num_access_files               0.005220
su_attempted                   0.004005
